# SLD pipeline inference: final models on 10 unseen test sheets

Loads the **final** detector and classifier from `sld_pipeline_artifacts.zip` (produced by `kaggle-sld-pipeline.ipynb`) and re-runs detect → trace wires → re-label on 10 sheets from the test split. No training here — this notebook only loads and scores.

**Before running**: GPU on, Internet on, attach two datasets: the sheet dataset (`data.yaml`, `images/test`, `graphs/`) and `sld_pipeline_artifacts.zip` uploaded as its own Kaggle dataset.

In [ ]:
import json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

from PIL import Image

In [ ]:
REPO_URL = "https://github.com/ishuu19/Single-Line-Diagram-and-Energy-Mapping.git"
REPO = Path("/kaggle/working/Single-Line-Diagram-and-Energy-Mapping")
COMP_DIR = REPO / "src" / "Object Detection" / "Component Training"
SLD_DIR = REPO / "src" / "Object Detection" / "SLD Training"
ARTIFACTS_DIR = Path("/kaggle/working/sld_pipeline_artifacts")
INPUT = Path("/kaggle/input")
N_SHEETS = 10

## Repo + dependencies

In [ ]:
def sh(*cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    return (r.stdout + r.stderr).strip()


if (REPO / ".git").is_dir():
    sh("git", "pull", "--ff-only", cwd=REPO)
else:
    sh("git", "clone", "-q", REPO_URL, str(REPO))
assert (SLD_DIR / "predict_graph.py").exists()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "scikit-image", "networkx"], check=True);

## Unpack `sld_pipeline_artifacts.zip` (the final models)

In [ ]:
def find_zip(name):
    hits = list(INPUT.rglob(name))
    return hits[0] if hits else None


zip_path = find_zip("sld_pipeline_artifacts.zip")
assert zip_path, "attach sld_pipeline_artifacts.zip as a Kaggle dataset"
if ARTIFACTS_DIR.exists():
    shutil.rmtree(ARTIFACTS_DIR)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(ARTIFACTS_DIR)

(COMP_DIR / "models").mkdir(parents=True, exist_ok=True)
(SLD_DIR / "models").mkdir(parents=True, exist_ok=True)
for f in (ARTIFACTS_DIR / "models" / "component").glob("*"):
    shutil.copy(f, COMP_DIR / "models" / f.name)
for f in (ARTIFACTS_DIR / "models" / "sld").glob("*"):
    shutil.copy(f, SLD_DIR / "models" / f.name)
sorted(p.name for p in (SLD_DIR / "models").glob("*")), sorted(p.name for p in (COMP_DIR / "models").glob("*"))

## Attach the sheet dataset, pick 10 test sheets

In [ ]:
os.chdir(SLD_DIR)
sys.path.insert(0, str(SLD_DIR))
import config
from sld_data import load_classes, sheet_ids, image_path, load_graph
from predict_graph import load_detector, predict_sheet
from reclassify import load_classifier, reclassify
from graph_eval import score_sheet, summarise
from viz import show_graph

SHEETS = next(p.parent.parent for p in INPUT.rglob("data.yaml") if (p.parent.parent / "graphs").is_dir())
if config.SHEETS_DIR.is_symlink() or config.SHEETS_DIR.exists():
    (config.SHEETS_DIR.unlink() if config.SHEETS_DIR.is_symlink() else shutil.rmtree(config.SHEETS_DIR))
config.SHEETS_DIR.parent.mkdir(parents=True, exist_ok=True)
os.symlink(SHEETS, config.SHEETS_DIR, target_is_directory=True)

classes = load_classes()
test_ids = sheet_ids("test")[:N_SHEETS]
len(classes), test_ids

## Load the final models

In [ ]:
detector = load_detector(SLD_DIR / "models" / "sld_yolov8s.pt")
classifier = load_classifier(COMP_DIR / "models" / "custom_best.keras")

## Detect → trace wires → re-label, then score against ground truth

In [ ]:
def recover(sid):
    img = Image.open(image_path(sid, "test"))
    result, graph = predict_sheet(detector, img, classes, sid)
    reclassify(img, result["symbols"], classifier)
    return img, result


recovered = {sid: recover(sid) for sid in test_ids}
rows = [score_sheet(load_graph(sid), recovered[sid][1]) for sid in test_ids]

## Accuracy on these 10 sheets

In [ ]:
print(json.dumps(summarise(rows), indent=2))

## Detected + traced graph on each of the 10 sheets

In [ ]:
for sid in test_ids:
    img, result = recovered[sid]
    show_graph(img, result, sid)